# 5 Noisy Data Analysis

This notebook analyzes the robustness of the neural network SHO fits against noise by
comparing them with least-squares fits (LSQF) at noise levels 0-8.

Notes:
- This notebook consumes the batch-training outputs produced by notebook
  `2_5_nn_fitting_all.ipynb` (run it first).
- Noise levels 1-8 additionally require the noisy datasets *and their LSQF SHO fits*
  generated by running notebook `0_5_Noisy_Data_and_Fitting.ipynb` at full scale, and
  the corresponding trained models from a full run of notebook 2_5 (designed for a GPU,
  e.g. Google Colab).
- With `QUICK_RUN = True` only the noise level 0 analysis is verified on a CPU using
  whatever models a `QUICK_RUN` of notebook 2_5 produced; the heaviest figures (the SHO
  fit movies) and all noise 1-8 sections are skipped.


In [ ]:
# --- Environment setup (works on Colab and locally) ---
REPO_URL = "https://github.com/m3-learning/m3_learning.git"
BRANCH = "shofit"
QUICK_RUN = False  # True = noise level 0 only, for fast verification

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import importlib.util, subprocess
    if importlib.util.find_spec("m3_learning") is None:
        subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1",
                        REPO_URL, "/content/m3_learning_repo"], check=True)
        subprocess.run(["pip", "install", "-q",
                        "/content/m3_learning_repo/m3_learning"], check=True)
        subprocess.run(["pip", "install", "-q", "--upgrade", "numpy_groupies"], check=True)
    subprocess.run(["pip", "install", "-q", "dataerai-cli", "dataerai-sdk[ml]", "keyring"], check=True)

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Colab: {IN_COLAB}, device: {device}")


## Dataerai login and provenance logging

Run these commands once before enabling lineage logging. On macOS, `dataerai auth login` is all you need: credentials are read from the system keychain via the `keyring` package. On Colab or headless machines, use device login.

```bash
python -m pip install dataerai-cli 'dataerai-sdk[ml]' keyring
dataerai auth login --device --server https://beta.dataerai.com
dataerai auth status
dataerai upload --project <project-uuid> --collection <collection-uuid> --metadata dataerai_source_asset.json --file Data/data_raw.h5
export DATAERAI_DATASET_ASSET_ID=<asset-id>
export DATAERAI_LOG_PROVENANCE=1
```

Leave `DATAERAI_LOG_PROVENANCE` unset to run without network logging. If you already know the provenance surrogate, set `DATAERAI_DATASET_RECORD_SK` instead of `DATAERAI_DATASET_ASSET_ID`.


In [ ]:
import os

DATAERAI_NOTEBOOK = '4_Noisy_Data_Analysis.ipynb'
LOG_DATAERAI_PROVENANCE = os.getenv("DATAERAI_LOG_PROVENANCE", "").lower() in {"1", "true", "yes", "on"}
DATAERAI_DATASET_ASSET_ID = os.getenv("DATAERAI_DATASET_ASSET_ID") or None
DATAERAI_DATASET_RECORD_SK = os.getenv("DATAERAI_DATASET_RECORD_SK") or None

if DATAERAI_DATASET_ASSET_ID or DATAERAI_DATASET_RECORD_SK:
    LOG_DATAERAI_PROVENANCE = True

print("Dataerai provenance logging:", "enabled" if LOG_DATAERAI_PROVENANCE else "disabled")


In [ ]:
%load_ext autoreload
%autoreload 2

import glob

import numpy as np

from m3_learning.nn.random import random_seed
from m3_learning.viz.style import set_style
from m3_learning.util.file_IO import download_and_unzip
from m3_learning.viz.printing import printer
from m3_learning.be.viz import Viz
from m3_learning.be.dataset import BE_Dataset
from m3_learning.be.nn import SHO_fit_func_nn, find_best_model
from m3_learning.nn.Fitter1D.Fitter1D import Multiscale1DFitter, Model, ComplexPostProcessor

printing = printer(basepath = './Figures/')


set_style("printing")
random_seed(seed=42)

%matplotlib inline


## Loads Data


In [ ]:
# Download the data file from Zenodo (skipped if the file already exists,
# e.g. if you already ran notebooks 0_5, 1, 2, 2_5 or 3)
url = 'https://zenodo.org/record/7774788/files/PZT_2080_raw_data.h5?download=1'

# Specify the filename and the path to save the file
filename = 'data_raw.h5'
save_path = './Data'

# download the file
download_and_unzip(filename, url, save_path)

data_path = save_path + "/" + filename

# instantiate the dataset object
dataset = BE_Dataset(data_path, SHO_fit_func_LSQF=SHO_fit_func_nn)

# computes the SHO LSQF fit if it has not been performed previously.
# This is cached/idempotent: it returns instantly if you already ran notebook 1 or 2.
dataset.SHO_Fitter(force=False, h5_sho_targ_grp="Raw_Data_SHO_Fit")

# reinstantiate the dataset object after fitting
dataset = BE_Dataset(data_path, SHO_fit_func_LSQF=SHO_fit_func_nn)

# print the contents of the file
dataset.print_be_tree()


# Benchmarking on Noisy Data

To benchmark on noisy data we conducted fits using both Adam and Trust Region Optimizers. We added noise in multiples of the standard deviation of the raw data. 

Training was saved after 900 seconds.

In [ ]:
# most recent batch-training output from notebook 2_5
candidates = sorted(glob.glob("Trained Models/SHO Fitter/*_nn_benchmarks_noise"))
assert candidates, "No batch-training outputs found - run notebook 2_5_nn_fitting_all.ipynb first."

basepath = candidates[-1]
csv_filename = "Batch_Trainging_SpeedTest.csv"

# finds the model with the lowest training loss for every (noise level, optimizer) pair
results = find_best_model(basepath, csv_filename)

print(f"Using batch-training results from: {basepath}")
print(f"Available (noise level, optimizer) combinations: {sorted(results.keys())}")


def load_nn_model(checkpoint_path):
    """Loads a trained SHO fitting neural network (the architecture trained by
    `batch_training` in notebook 2_5) from a saved checkpoint."""
    fitter = Multiscale1DFitter(SHO_fit_func_nn,  # function
                                dataset.frequency_bin,  # x data
                                2,  # input channels
                                4,  # output parameters
                                dataset.SHO_scaler,
                                ComplexPostProcessor(dataset))
    model = Model(fitter, dataset, training=False,
                  model_basename="SHO_Fitter_original_data")
    model.load(checkpoint_path)
    return model


## Instantiate the Visualizer

In [ ]:
# insatiate the visualization object
image_scalebar = [2000, 500, "nm", "br"]

BE_viz = Viz(dataset, printing, verbose=True, 
             SHO_ranges = [(0,1.5e-4), (1.31e6, 1.33e6), (-300, 0), (-np.pi, np.pi)],
             image_scalebar=image_scalebar)

# extracts the x and y data based on the noise
X_data_no_noise, Y_data = dataset.NN_data()

## Noise Level 0, ADAM Optimizer

### Scaling the Data

When training the neural network it is useful to scale the data. We apply a global scaler such that the spectrum have a mean of 0 and a standard deviation of 1.

#### Visualizing the Scaled Data


In [ ]:
# Sets the dataset
noise = 0
optimizer = "Adam"

state = {"fitter": "LSQF", "resampled": True, "scaled": True, "label": "Scaled", "noise": noise}
dataset.set_attributes(**state)

BE_viz.nn_checker(state, filename=f"Figure_5_1_Scaled Raw Data_noise{noise}_optimizer_{optimizer}")

**Figure 5.1** Example visualization of the scaled, noisy data which is used for training.

### Extracts the Data and Models

In [ ]:
# extracts the x and y data based on the noise
X_data, Y_data = dataset.NN_data()

# searches the trained models for the best model and loads it
# (same architecture as trained by notebook 2_5)
model_name_adam = basepath + "/" + results[(noise, "Adam")]['filename'].split("//")[-1]
model_adam = load_nn_model(model_name_adam)

# a QUICK_RUN of notebook 2_5 only trains an Adam model on the raw data; the
# Trust-Region comparisons below are skipped when no such model is available
if (noise, "Trust Region CG") in results:
    model_name_trust_region = basepath + "/" + results[(noise, "Trust Region CG")]['filename'].split("//")[-1]
    model_trust_region = load_nn_model(model_name_trust_region)
else:
    model_trust_region = None
    print("QUICK_RUN: no Trust Region CG model available — requires full 0_5/2_5 outputs — run on Colab at full scale")


### Evaluate the Fit Results

It is always recommended to validate that the autoencoder is working correctly. We can do this by visualizing the best, median, and worst fits.

We will assume that the autoencoder is working correctly and thus will not consider the test train split.

Note: we are comparing the autoencoder results to the original data, not the noisy data. 


In [ ]:
LSQF_ = {'resampled': True,
                'raw_format': 'complex',
                'fitter': 'LSQF',
                'scaled': True,
                'output_shape': 'index',
                'measurement_state': 'all',
                'resampled_bins': 165,
                'LSQF_phase_shift': 1.5707963267948966,
                'NN_phase_shift': None,
                'noise': noise}

# builds the list of predictions to compare; the Trust-Region model is only
# included when it is available (i.e., after a full run of notebook 2_5)
predictions_ = [model_adam, model_trust_region, LSQF_]
labels_ = ["Adam", "Trust Region", "LSQF"]
predictions_, labels_ = map(list, zip(
    *[(p, l) for p, l in zip(predictions_, labels_) if p is not None]))

if QUICK_RUN:
    # the LSQF reconstruction always spans the full dataset, which dominates the
    # CPU runtime; under QUICK_RUN the MSE comparison uses the NN models on a subset
    labels_ = [l for p, l in zip(predictions_, labels_) if not isinstance(p, dict)]
    predictions_ = [p for p in predictions_ if not isinstance(p, dict)]
    BE_viz.MSE_compare(X_data_no_noise[:10000], predictions_, labels_)
else:
    BE_viz.MSE_compare(X_data_no_noise, predictions_, labels_)


#### Least Squares Fit

In [ ]:

d1, d2, index1, mse1 = BE_viz.bmw_nn(
    X_data,
    prediction=LSQF_,
    out_state={"scaled": True, "raw_format": "complex"},
    returns=True,
    filename=f"Figure_5_2_NN_validation_noise_{noise}",
    compare_state = X_data_no_noise,
)

**Figure 5.2** Visualization of the noisy (noise level 0) fit results from the least squares fitting algorithm shows the best, median, and worst fits.

#### Neural Network with Adam Optimizer

In [ ]:
# under QUICK_RUN, validate on a subset of the data to keep the CPU runtime low
X_eval = X_data[:10000] if QUICK_RUN else X_data
X_compare = X_data_no_noise[:10000] if QUICK_RUN else X_data_no_noise

d1, d2, index1, mse1 = BE_viz.bmw_nn(
    X_eval,
    prediction=model_adam,
    out_state={"scaled": True, "raw_format": "complex"},
    returns=True,
    filename=f"Figure_5_3_NN_validation_noise_{noise}_Adam",
    compare_state = X_compare,
)


**Figure 5.3** Visualization of the noisy (noise level 0) fit results from the neural network trained with the Adam optimizers. We shows the best, median, and worst fits.

##### Neural Network with Trust Region Conjugate Gradient Optimizer

In [ ]:
if model_trust_region is None:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    d1, d2, index1, mse1 = BE_viz.bmw_nn(
        X_data,
        prediction=model_trust_region,
        out_state={"scaled": True, "raw_format": "complex"},
        returns=True,
        filename=f"Figure_5_4_NN_validation_noise_{noise}_Trust_Region",
        compare_state = X_data_no_noise,
    )


**Figure 5.4** Visualization of the noisy fit results from the neural network trained with the Adam optimizers. We shows the best, median, and worst fits.

### Histogram of Fit Results

It is useful to view the histogram of the fitting results to apply any necessary phase shifts, and to see if the results are reasonable.


In [ ]:
# make all the histograms the same bin range.

LSQF_ = {'resampled': True,
                'raw_format': 'complex',
                'fitter': 'LSQF',
                'scaled': False,
                'output_shape': 'index',
                'measurement_state': 'all',
                'resampled_bins': 165,
                'LSQF_phase_shift': 1.5707963267948966,
                'NN_phase_shift': None,
                'noise': noise}

dataset.set_attributes(**LSQF_)

LSQF_params = dataset.SHO_fit_results(state = LSQF_)

# X_data is passed explicitly to avoid re-extracting the (identical) NN input data
adam_params = dataset.SHO_fit_results(model = model_adam, phase_shift = np.pi / 2, X_data = X_data)

if model_trust_region is not None:
    trust_region_params = dataset.SHO_fit_results(model = model_trust_region, phase_shift = np.pi / 2, X_data = X_data)
else:
    trust_region_params = None


In [ ]:
params_ = [p for p in [LSQF_params, adam_params, trust_region_params] if p is not None]

BE_viz.SHO_hist(params_, filename=f"Figure_5_5_Histogram_comparison_{noise}_noise",)


**Figure 5.5** Histogram of the fit results for the a-d. LSQF, e-h. neural network with ADAM, i-l. neural network with trust region optimizers. 

In [ ]:
# reuses the SHO fit parameters computed above (identical states and models)
maps_params = [p for p in [LSQF_params, adam_params, trust_region_params] if p is not None]
maps_labels = ["LSQF", "Adam", "SGD"][:len(maps_params)]

BE_viz.SHO_switching_maps_test(maps_params, filename=f"Figure_5_6_switching_maps_comparison_{noise}_noise", labels=maps_labels)


**Figure 5.6** Visualize the switching based on the different models on the noisy data. 

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: skipping the SHO fit movie (CPU-heavy) — run on Colab at full scale")
else:
    BE_viz.SHO_fit_movie_images(noise = 0,
                                models = [None, model_adam, model_trust_region],
                                scalebar_= True,
                                basepath = "Movies/SHO_NN_compare",
                                filename="SHO_NN_compare",
                                phase_shift = [None, np.pi / 2, np.pi / 2])


## Noise Level 1

### Scaling the Data

When training the neural network it is useful to scale the data. We apply a global scaler such that the spectrum have a mean of 0 and a standard deviation of 1.

#### Visualizing the Scaled Data


In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    # Sets the dataset
    noise = 1
    optimizer = "Adam"

    state = {"fitter": "LSQF", "resampled": True, "scaled": True, "label": "Scaled", "noise": noise}
    dataset.set_attributes(**state)

    BE_viz.nn_checker(state, filename=f"Figure_5_7_Scaled Raw Data_noise{noise}_optimizer_{optimizer}")

**Figure 5.7** Example visualization of the scaled, noisy data which is used for training.

### Extracts the Data and Models

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    # extracts the x and y data based on the noise
    X_data, Y_data = dataset.NN_data()

    # searches the trained models for the best model and loads it
    # (same architecture as trained by notebook 2_5)
    model_name_adam = basepath + "/" + results[(noise, "Adam")]['filename'].split("//")[-1]
    model_name_trust_region = basepath + "/" + results[(noise, "Trust Region CG")]['filename'].split("//")[-1]

    model_adam = load_nn_model(model_name_adam)

    model_trust_region = load_nn_model(model_name_trust_region)

### Evaluate the Fit Results

It is always recommended to validate that the autoencoder is working correctly. We can do this by visualizing the best, median, and worst fits.

We will assume that the autoencoder is working correctly and thus will not consider the test train split.

Note: we are comparing the autoencoder results to the original data, not the noisy data. 


In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    LSQF_ = {'resampled': True,
                    'raw_format': 'complex',
                    'fitter': 'LSQF',
                    'scaled': True,
                    'output_shape': 'index',
                    'measurement_state': 'all',
                    'resampled_bins': 165,
                    'LSQF_phase_shift': 1.5707963267948966,
                    'NN_phase_shift': None,
                    'noise': noise}


    BE_viz.MSE_compare(X_data_no_noise, [model_adam, model_trust_region, LSQF_], ["Adam", "Trust Region", "LSQF"])

#### Least Squares Fit

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:

    d1, d2, index1, mse1 = BE_viz.bmw_nn(
        X_data,
        prediction=LSQF_,
        out_state={"scaled": True, "raw_format": "complex"},
        returns=True,
        filename=f"Figure_5_8_NN_validation_noise_{noise}",
        compare_state = X_data_no_noise,
    )

**Figure 5.8** Visualization of the noisy (noise level 1) fit results from the least squares fitting algorithm shows the best, median, and worst fits.

#### Neural Network with Adam Optimizer

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    d1, d2, index1, mse1 = BE_viz.bmw_nn(
        X_data,
        prediction=model_adam,
        out_state={"scaled": True, "raw_format": "complex"},
        returns=True,
        filename=f"Figure_5_9_NN_validation_noise_{noise}_Adam",
        compare_state = X_data_no_noise,
    )

**Figure 5.9** Visualization of the noisy (noise level 0) fit results from the neural network trained with the Adam optimizers. We shows the best, median, and worst fits.

##### Neural Network with Trust Region Conjugate Gradient Optimizer

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    d1, d2, index1, mse1 = BE_viz.bmw_nn(
        X_data,
        prediction=model_trust_region,
        out_state={"scaled": True, "raw_format": "complex"},
        returns=True,
        filename=f"Figure_5_10_NN_validation_noise_{noise}_Trust_Region",
        compare_state = X_data_no_noise,
    )

**Figure 5.10** Visualization of the noisy fit results from the neural network trained with the Adam optimizers. We shows the best, median, and worst fits.

### Histogram of Fit Results

It is useful to view the histogram of the fitting results to apply any necessary phase shifts, and to see if the results are reasonable.


In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    # make all the histograms the same bin range.

    LSQF_ = {'resampled': True,
                    'raw_format': 'complex',
                    'fitter': 'LSQF',
                    'scaled': False,
                    'output_shape': 'index',
                    'measurement_state': 'all',
                    'resampled_bins': 165,
                    'LSQF_phase_shift': 1.5707963267948966,
                    'NN_phase_shift': None,
                    'noise': noise}

    dataset.set_attributes(**LSQF_)

    LSQF_params = dataset.SHO_fit_results(state = LSQF_)

    # adam_params = dataset.SHO_fit_results(model = model_adam, phase_shift = np.pi / 2)

    # trust_region_params = dataset.SHO_fit_results(model = model_trust_region, phase_shift = np.pi / 2)

    adam_params = dataset.SHO_fit_results(model = model_adam, phase_shift = np.pi / 2)

    trust_region_params = dataset.SHO_fit_results(model = model_trust_region, phase_shift = np.pi / 2)

    #BE_viz.SHO_hist([LSQF_params, adam_params, trust_region_params], SHO_ranges = BE_viz.SHO_ranges, filename=f"Figure_5_11_Histogram_comparison_{noise}_noise",)
    BE_viz.SHO_hist([LSQF_params, adam_params, trust_region_params], filename=f"Figure_5_11_Histogram_comparison_{noise}_noise",)

**Figure 5.11** Histogram of the fit results based on fits with noise 1 for the a-d. LSQF, e-h. neural network with ADAM, i-l. neural network with trust region optimizers.

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    LSQF_ = {'resampled': True,
                    'raw_format': 'complex',
                    'fitter': 'LSQF',
                    'scaled': False,
                    'output_shape': 'index',
                    'measurement_state': 'all',
                    'resampled_bins': 165,
                    'LSQF_phase_shift': 1.5707963267948966,
                    'NN_phase_shift': None,
                    'noise': noise}

    LSQF_Params = dataset.SHO_fit_results(state = LSQF_)

    adam_params = dataset.SHO_fit_results(model = model_adam, phase_shift = np.pi / 2)

    trust_region_params = dataset.SHO_fit_results(model = model_trust_region, phase_shift = np.pi / 2)

    BE_viz.SHO_switching_maps_test([LSQF_Params, adam_params, trust_region_params], filename=f"Figure_5_12_switching_maps_comparison_{noise}_noise", labels=["LSQF", "Adam", "SGD"])

**Figure 5.12** Visualize the switching based on the different models on the noisy data. 

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    BE_viz.SHO_fit_movie_images(noise = noise, 
                                models = [None, model_adam, model_trust_region],
                                scalebar_= True, 
                                basepath = "Movies/SHO_NN_compare",  
                                filename="SHO_NN_compare",
                                phase_shift = [None, np.pi / 2, np.pi / 2])

## Noise Level 2

### Scaling the Data

When training the neural network it is useful to scale the data. We apply a global scaler such that the spectrum have a mean of 0 and a standard deviation of 1.

#### Visualizing the Scaled Data


In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    # Sets the dataset
    noise = 2
    optimizer = "Adam"

    state = {"fitter": "LSQF", "resampled": True, "scaled": True, "label": "Scaled", "noise": noise}
    dataset.set_attributes(**state)

    BE_viz.nn_checker(state, filename=f"Figure_5_13_Scaled Raw Data_noise{noise}_optimizer_{optimizer}")

**Figure 5.13** Example visualization of the scaled, noisy data which is used for training.

### Extracts the Data and Models

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    # extracts the x and y data based on the noise
    X_data, Y_data = dataset.NN_data()

    # searches the trained models for the best model and loads it
    # (same architecture as trained by notebook 2_5)
    model_name_adam = basepath + "/" + results[(noise, "Adam")]['filename'].split("//")[-1]
    model_name_trust_region = basepath + "/" + results[(noise, "Trust Region CG")]['filename'].split("//")[-1]

    model_adam = load_nn_model(model_name_adam)

    model_trust_region = load_nn_model(model_name_trust_region)

### Evaluate the Fit Results

It is always recommended to validate that the autoencoder is working correctly. We can do this by visualizing the best, median, and worst fits.

We will assume that the autoencoder is working correctly and thus will not consider the test train split.

Note: we are comparing the autoencoder results to the original data, not the noisy data. 


In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    LSQF_ = {'resampled': True,
                    'raw_format': 'complex',
                    'fitter': 'LSQF',
                    'scaled': True,
                    'output_shape': 'index',
                    'measurement_state': 'all',
                    'resampled_bins': 165,
                    'LSQF_phase_shift': 1.5707963267948966,
                    'NN_phase_shift': None,
                    'noise': noise}


    BE_viz.MSE_compare(X_data_no_noise, [model_adam, model_trust_region, LSQF_], ["Adam", "Trust Region", "LSQF"])

#### Least Squares Fit

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:

    d1, d2, index1, mse1 = BE_viz.bmw_nn(
        X_data,
        prediction=LSQF_,
        out_state={"scaled": True, "raw_format": "complex"},
        returns=True,
        filename=f"Figure_5_14_NN_validation_noise_{noise}",
        compare_state = X_data_no_noise,
    )

**Figure 5.14** Visualization of the noisy (noise level 2) fit results from the least squares fitting algorithm shows the best, median, and worst fits.

#### Neural Network with Adam Optimizer

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    d1, d2, index1, mse1 = BE_viz.bmw_nn(
        X_data,
        prediction=model_adam,
        out_state={"scaled": True, "raw_format": "complex"},
        returns=True,
        filename=f"Figure_5_15_NN_validation_noise_{noise}_Adam",
        compare_state = X_data_no_noise,
    )

**Figure 5.15** Visualization of the noisy (noise level 0) fit results from the neural network trained with the Adam optimizers. We shows the best, median, and worst fits.

##### Neural Network with Trust Region Conjugate Gradient Optimizer

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    d1, d2, index1, mse1 = BE_viz.bmw_nn(
        X_data,
        prediction=model_trust_region,
        out_state={"scaled": True, "raw_format": "complex"},
        returns=True,
        filename=f"Figure_5_16_NN_validation_noise_{noise}_Trust_Region",
        compare_state = X_data_no_noise,
    )

**Figure 5.16** Visualization of the noisy fit results from the neural network trained with the Adam optimizers. We shows the best, median, and worst fits.

### Histogram of Fit Results

It is useful to view the histogram of the fitting results to apply any necessary phase shifts, and to see if the results are reasonable.


In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    # make all the histograms the same bin range.

    LSQF_ = {'resampled': True,
                    'raw_format': 'complex',
                    'fitter': 'LSQF',
                    'scaled': False,
                    'output_shape': 'index',
                    'measurement_state': 'all',
                    'resampled_bins': 165,
                    'LSQF_phase_shift': 1.5707963267948966,
                    'NN_phase_shift': None,
                    'noise': noise}

    dataset.set_attributes(**LSQF_)

    LSQF_params = dataset.SHO_fit_results(state = LSQF_)

    adam_params = dataset.SHO_fit_results(model = model_adam, phase_shift = np.pi / 2)

    trust_region_params = dataset.SHO_fit_results(model = model_trust_region, phase_shift = np.pi / 2)

    #BE_viz.SHO_hist([LSQF_params, adam_params, trust_region_params], SHO_ranges = BE_viz.SHO_ranges, filename=f"Figure_5_16_Histogram_comparison_{noise}_noise",)
    BE_viz.SHO_hist([LSQF_params, adam_params, trust_region_params], filename=f"Figure_5_16_Histogram_comparison_{noise}_noise",)

**Figure 5.16** Histogram of the fit results based on fits with noise 1 for the a-d. LSQF, e-h. neural network with ADAM, i-l. neural network with trust region optimizers.

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    LSQF_ = {'resampled': True,
                    'raw_format': 'complex',
                    'fitter': 'LSQF',
                    'scaled': False,
                    'output_shape': 'index',
                    'measurement_state': 'all',
                    'resampled_bins': 165,
                    'LSQF_phase_shift': 1.5707963267948966,
                    'NN_phase_shift': None,
                    'noise': noise}

    LSQF_Params = dataset.SHO_fit_results(state = LSQF_)

    adam_params = dataset.SHO_fit_results(model = model_adam, phase_shift = np.pi / 2)

    trust_region_params = dataset.SHO_fit_results(model = model_trust_region, phase_shift = np.pi / 2)

    BE_viz.SHO_switching_maps_test([LSQF_Params, adam_params, trust_region_params], filename=f"Figure_5_17_switching_maps_comparison_{noise}_noise", labels=["LSQF", "Adam", "SGD"])

**Figure 5.17** Visualize the switching based on the different models on the noisy data. 

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    BE_viz.SHO_fit_movie_images(noise = noise, 
                                models = [None, model_adam, model_trust_region],
                                scalebar_= True, 
                                basepath = "Movies/SHO_NN_compare",  
                                filename="SHO_NN_compare",
                                phase_shift = [None, np.pi / 2, np.pi / 2])

## Noise Level 3

### Scaling the Data

When training the neural network it is useful to scale the data. We apply a global scaler such that the spectrum have a mean of 0 and a standard deviation of 1.

#### Visualizing the Scaled Data


In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    # Sets the dataset
    noise = 3
    optimizer = "Adam"

    state = {"fitter": "LSQF", "resampled": True, "scaled": True, "label": "Scaled", "noise": noise}
    dataset.set_attributes(**state)

    BE_viz.nn_checker(state, filename=f"Figure_5_18_Scaled Raw Data_noise{noise}_optimizer_{optimizer}")

**Figure 5.18** Example visualization of the scaled, noisy data which is used for training.

### Extracts the Data and Models

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    # extracts the x and y data based on the noise
    X_data, Y_data = dataset.NN_data()

    # searches the trained models for the best model and loads it
    # (same architecture as trained by notebook 2_5)
    model_name_adam = basepath + "/" + results[(noise, "Adam")]['filename'].split("//")[-1]
    model_name_trust_region = basepath + "/" + results[(noise, "Trust Region CG")]['filename'].split("//")[-1]

    model_adam = load_nn_model(model_name_adam)

    model_trust_region = load_nn_model(model_name_trust_region)

### Evaluate the Fit Results

It is always recommended to validate that the autoencoder is working correctly. We can do this by visualizing the best, median, and worst fits.

We will assume that the autoencoder is working correctly and thus will not consider the test train split.

Note: we are comparing the autoencoder results to the original data, not the noisy data. 


In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    LSQF_ = {'resampled': True,
                    'raw_format': 'complex',
                    'fitter': 'LSQF',
                    'scaled': True,
                    'output_shape': 'index',
                    'measurement_state': 'all',
                    'resampled_bins': 165,
                    'LSQF_phase_shift': 1.5707963267948966,
                    'NN_phase_shift': None,
                    'noise': noise}


    BE_viz.MSE_compare(X_data_no_noise, [model_adam, model_trust_region, LSQF_], ["Adam", "Trust Region", "LSQF"])

#### Least Squares Fit

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:

    d1, d2, index1, mse1 = BE_viz.bmw_nn(
        X_data,
        prediction=LSQF_,
        out_state={"scaled": True, "raw_format": "complex"},
        returns=True,
        filename=f"Figure_5_19_NN_validation_noise_{noise}",
        compare_state = X_data_no_noise,
    )

**Figure 5.19** Visualization of the noisy (noise level 1) fit results from the least squares fitting algorithm shows the best, median, and worst fits.

#### Neural Network with Adam Optimizer

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    d1, d2, index1, mse1 = BE_viz.bmw_nn(
        X_data,
        prediction=model_adam,
        out_state={"scaled": True, "raw_format": "complex"},
        returns=True,
        filename=f"Figure_5_20_NN_validation_noise_{noise}_Adam",
        compare_state = X_data_no_noise,
    )

**Figure 5.20** Visualization of the noisy (noise level 0) fit results from the neural network trained with the Adam optimizers. We shows the best, median, and worst fits.

##### Neural Network with Trust Region Conjugate Gradient Optimizer

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    d1, d2, index1, mse1 = BE_viz.bmw_nn(
        X_data,
        prediction=model_trust_region,
        out_state={"scaled": True, "raw_format": "complex"},
        returns=True,
        filename=f"Figure_5_21_NN_validation_noise_{noise}_Trust_Region",
        compare_state = X_data_no_noise,
    )

**Figure 5.21** Visualization of the noisy fit results from the neural network trained with the Adam optimizers. We shows the best, median, and worst fits.

### Histogram of Fit Results

It is useful to view the histogram of the fitting results to apply any necessary phase shifts, and to see if the results are reasonable.


In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    # make all the histograms the same bin range.

    LSQF_ = {'resampled': True,
                    'raw_format': 'complex',
                    'fitter': 'LSQF',
                    'scaled': False,
                    'output_shape': 'index',
                    'measurement_state': 'all',
                    'resampled_bins': 165,
                    'LSQF_phase_shift': 1.5707963267948966,
                    'NN_phase_shift': None,
                    'noise': noise}

    dataset.set_attributes(**LSQF_)

    LSQF_params = dataset.SHO_fit_results(state = LSQF_)

    adam_params = dataset.SHO_fit_results(model = model_adam, phase_shift = np.pi / 2)

    trust_region_params = dataset.SHO_fit_results(model = model_trust_region, phase_shift = np.pi / 2)

    #BE_viz.SHO_hist([LSQF_params, adam_params, trust_region_params], SHO_ranges = BE_viz.SHO_ranges, filename=f"Figure_5_22_Histogram_comparison_{noise}_noise",)
    BE_viz.SHO_hist([LSQF_params, adam_params, trust_region_params], filename=f"Figure_5_22_Histogram_comparison_{noise}_noise",)

**Figure 5.22** Histogram of the fit results based on fits with noise 1 for the a-d. LSQF, e-h. neural network with ADAM, i-l. neural network with trust region optimizers.

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    LSQF_ = {'resampled': True,
                    'raw_format': 'complex',
                    'fitter': 'LSQF',
                    'scaled': False,
                    'output_shape': 'index',
                    'measurement_state': 'all',
                    'resampled_bins': 165,
                    'LSQF_phase_shift': 1.5707963267948966,
                    'NN_phase_shift': None,
                    'noise': noise}

    LSQF_Params = dataset.SHO_fit_results(state = LSQF_)

    adam_params = dataset.SHO_fit_results(model = model_adam, phase_shift = np.pi / 2)

    trust_region_params = dataset.SHO_fit_results(model = model_trust_region, phase_shift = np.pi / 2)

    BE_viz.SHO_switching_maps_test([LSQF_Params, adam_params, trust_region_params], filename=f"Figure_5_23_switching_maps_comparison_{noise}_noise", labels=["LSQF", "Adam", "SGD"])

**Figure 5.23** Visualize the switching based on the different models on the noisy data. 

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    BE_viz.SHO_fit_movie_images(noise = noise, 
                                models = [None, model_adam, model_trust_region],
                                scalebar_= True, 
                                basepath = "Movies/SHO_NN_compare",  
                                filename="SHO_NN_compare",
                                phase_shift = [None, np.pi / 2, np.pi / 2])

## Noise Level 4

### Scaling the Data

When training the neural network it is useful to scale the data. We apply a global scaler such that the spectrum have a mean of 0 and a standard deviation of 1.

#### Visualizing the Scaled Data


In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    # Sets the dataset
    noise = 4
    optimizer = "Adam"

    state = {"fitter": "LSQF", "resampled": True, "scaled": True, "label": "Scaled", "noise": noise}
    dataset.set_attributes(**state)

    BE_viz.nn_checker(state, filename=f"Figure_5_24_Scaled Raw Data_noise{noise}_optimizer_{optimizer}")

**Figure 5.24** Example visualization of the scaled, noisy data which is used for training.

### Extracts the Data and Models

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    # extracts the x and y data based on the noise
    X_data, Y_data = dataset.NN_data()

    # searches the trained models for the best model and loads it
    # (same architecture as trained by notebook 2_5)
    model_name_adam = basepath + "/" + results[(noise, "Adam")]['filename'].split("//")[-1]
    model_name_trust_region = basepath + "/" + results[(noise, "Trust Region CG")]['filename'].split("//")[-1]

    model_adam = load_nn_model(model_name_adam)

    model_trust_region = load_nn_model(model_name_trust_region)

### Evaluate the Fit Results

It is always recommended to validate that the autoencoder is working correctly. We can do this by visualizing the best, median, and worst fits.

We will assume that the autoencoder is working correctly and thus will not consider the test train split.

Note: we are comparing the autoencoder results to the original data, not the noisy data. 


In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    LSQF_ = {'resampled': True,
                    'raw_format': 'complex',
                    'fitter': 'LSQF',
                    'scaled': True,
                    'output_shape': 'index',
                    'measurement_state': 'all',
                    'resampled_bins': 165,
                    'LSQF_phase_shift': 1.5707963267948966,
                    'NN_phase_shift': None,
                    'noise': noise}


    BE_viz.MSE_compare(X_data_no_noise, [model_adam, model_trust_region, LSQF_], ["Adam", "Trust Region", "LSQF"])

#### Least Squares Fit

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:

    d1, d2, index1, mse1 = BE_viz.bmw_nn(
        X_data,
        prediction=LSQF_,
        out_state={"scaled": True, "raw_format": "complex"},
        returns=True,
        filename=f"Figure_5_25_NN_validation_noise_{noise}",
        compare_state = X_data_no_noise,
    )

**Figure 5.25** Visualization of the noisy (noise level 1) fit results from the least squares fitting algorithm shows the best, median, and worst fits.

#### Neural Network with Adam Optimizer

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    d1, d2, index1, mse1 = BE_viz.bmw_nn(
        X_data,
        prediction=model_adam,
        out_state={"scaled": True, "raw_format": "complex"},
        returns=True,
        filename=f"Figure_5_26_NN_validation_noise_{noise}_Adam",
        compare_state = X_data_no_noise,
    )

**Figure 5.26** Visualization of the noisy (noise level 0) fit results from the neural network trained with the Adam optimizers. We shows the best, median, and worst fits.

##### Neural Network with Trust Region Conjugate Gradient Optimizer

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    d1, d2, index1, mse1 = BE_viz.bmw_nn(
        X_data,
        prediction=model_trust_region,
        out_state={"scaled": True, "raw_format": "complex"},
        returns=True,
        filename=f"Figure_5_27_NN_validation_noise_{noise}_Trust_Region",
        compare_state = X_data_no_noise,
    )

**Figure 5.27** Visualization of the noisy fit results from the neural network trained with the Adam optimizers. We shows the best, median, and worst fits.

### Histogram of Fit Results

It is useful to view the histogram of the fitting results to apply any necessary phase shifts, and to see if the results are reasonable.


In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    # make all the histograms the same bin range.

    LSQF_ = {'resampled': True,
                    'raw_format': 'complex',
                    'fitter': 'LSQF',
                    'scaled': False,
                    'output_shape': 'index',
                    'measurement_state': 'all',
                    'resampled_bins': 165,
                    'LSQF_phase_shift': 1.5707963267948966,
                    'NN_phase_shift': None,
                    'noise': noise}

    dataset.set_attributes(**LSQF_)

    LSQF_params = dataset.SHO_fit_results(state = LSQF_)

    # adam_params = dataset.SHO_fit_results(model = model_adam, phase_shift = np.pi / 2)

    # trust_region_params = dataset.SHO_fit_results(model = model_trust_region, phase_shift = np.pi / 2)

    adam_params = dataset.SHO_fit_results(model = model_adam, phase_shift = np.pi / 2)

    trust_region_params = dataset.SHO_fit_results(model = model_trust_region, phase_shift = np.pi / 2)

    #BE_viz.SHO_hist([LSQF_params, adam_params, trust_region_params], SHO_ranges = BE_viz.SHO_ranges, filename=f"Figure_5_28_Histogram_comparison_{noise}_noise",)
    BE_viz.SHO_hist([LSQF_params, adam_params, trust_region_params], filename=f"Figure_5_28_Histogram_comparison_{noise}_noise",)

**Figure 5.28** Histogram of the fit results based on fits with noise 1 for the a-d. LSQF, e-h. neural network with ADAM, i-l. neural network with trust region optimizers.

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    LSQF_ = {'resampled': True,
                    'raw_format': 'complex',
                    'fitter': 'LSQF',
                    'scaled': False,
                    'output_shape': 'index',
                    'measurement_state': 'all',
                    'resampled_bins': 165,
                    'LSQF_phase_shift': 1.5707963267948966,
                    'NN_phase_shift': None,
                    'noise': noise}

    LSQF_Params = dataset.SHO_fit_results(state = LSQF_)

    adam_params = dataset.SHO_fit_results(model = model_adam, phase_shift = np.pi / 2)

    trust_region_params = dataset.SHO_fit_results(model = model_trust_region, phase_shift = np.pi / 2)

    BE_viz.SHO_switching_maps_test([LSQF_Params, adam_params, trust_region_params], filename=f"Figure_5_29_switching_maps_comparison_{noise}_noise", labels=["LSQF", "Adam", "SGD"])

**Figure 5.29** Visualize the switching based on the different models on the noisy data. 

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    BE_viz.SHO_fit_movie_images(noise = noise, 
                                models = [None, model_adam, model_trust_region],
                                scalebar_= True, 
                                basepath = "Movies/SHO_NN_compare",  
                                filename="SHO_NN_compare",
                                phase_shift = [None, np.pi / 2, np.pi / 2])

## Noise Level 5

### Scaling the Data

When training the neural network it is useful to scale the data. We apply a global scaler such that the spectrum have a mean of 0 and a standard deviation of 1.

#### Visualizing the Scaled Data


In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    # Sets the dataset
    noise = 5
    optimizer = "Adam"

    state = {"fitter": "LSQF", "resampled": True, "scaled": True, "label": "Scaled", "noise": noise}
    dataset.set_attributes(**state)

    BE_viz.nn_checker(state, filename=f"Figure_5_30_Scaled Raw Data_noise{noise}_optimizer_{optimizer}")

**Figure 5.30** Example visualization of the scaled, noisy data which is used for training.

### Extracts the Data and Models

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    # extracts the x and y data based on the noise
    X_data, Y_data = dataset.NN_data()

    # searches the trained models for the best model and loads it
    # (same architecture as trained by notebook 2_5)
    model_name_adam = basepath + "/" + results[(noise, "Adam")]['filename'].split("//")[-1]
    model_name_trust_region = basepath + "/" + results[(noise, "Trust Region CG")]['filename'].split("//")[-1]

    model_adam = load_nn_model(model_name_adam)

    model_trust_region = load_nn_model(model_name_trust_region)

### Evaluate the Fit Results

It is always recommended to validate that the autoencoder is working correctly. We can do this by visualizing the best, median, and worst fits.

We will assume that the autoencoder is working correctly and thus will not consider the test train split.

Note: we are comparing the autoencoder results to the original data, not the noisy data. 


In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    LSQF_ = {'resampled': True,
                    'raw_format': 'complex',
                    'fitter': 'LSQF',
                    'scaled': True,
                    'output_shape': 'index',
                    'measurement_state': 'all',
                    'resampled_bins': 165,
                    'LSQF_phase_shift': 1.5707963267948966,
                    'NN_phase_shift': None,
                    'noise': noise}


    BE_viz.MSE_compare(X_data_no_noise, [model_adam, model_trust_region, LSQF_], ["Adam", "Trust Region", "LSQF"])

#### Least Squares Fit

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:

    d1, d2, index1, mse1 = BE_viz.bmw_nn(
        X_data,
        prediction=LSQF_,
        out_state={"scaled": True, "raw_format": "complex"},
        returns=True,
        filename=f"Figure_5_31_NN_validation_noise_{noise}",
        compare_state = X_data_no_noise,
    )

**Figure 5.31** Visualization of the noisy (noise level 1) fit results from the least squares fitting algorithm shows the best, median, and worst fits.

#### Neural Network with Adam Optimizer

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    d1, d2, index1, mse1 = BE_viz.bmw_nn(
        X_data,
        prediction=model_adam,
        out_state={"scaled": True, "raw_format": "complex"},
        returns=True,
        filename=f"Figure_5_32_NN_validation_noise_{noise}_Adam",
        compare_state = X_data_no_noise,
    )

**Figure 5.32** Visualization of the noisy (noise level 0) fit results from the neural network trained with the Adam optimizers. We shows the best, median, and worst fits.

##### Neural Network with Trust Region Conjugate Gradient Optimizer

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    d1, d2, index1, mse1 = BE_viz.bmw_nn(
        X_data,
        prediction=model_trust_region,
        out_state={"scaled": True, "raw_format": "complex"},
        returns=True,
        filename=f"Figure_5_33_NN_validation_noise_{noise}_Trust_Region",
        compare_state = X_data_no_noise,
    )

**Figure 5.33** Visualization of the noisy fit results from the neural network trained with the Adam optimizers. We shows the best, median, and worst fits.

### Histogram of Fit Results

It is useful to view the histogram of the fitting results to apply any necessary phase shifts, and to see if the results are reasonable.


In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    # make all the histograms the same bin range.

    LSQF_ = {'resampled': True,
                    'raw_format': 'complex',
                    'fitter': 'LSQF',
                    'scaled': False,
                    'output_shape': 'index',
                    'measurement_state': 'all',
                    'resampled_bins': 165,
                    'LSQF_phase_shift': 1.5707963267948966,
                    'NN_phase_shift': None,
                    'noise': noise}

    dataset.set_attributes(**LSQF_)

    LSQF_params = dataset.SHO_fit_results(state = LSQF_)

    # adam_params = dataset.SHO_fit_results(model = model_adam, phase_shift = np.pi / 2)

    # trust_region_params = dataset.SHO_fit_results(model = model_trust_region, phase_shift = np.pi / 2)

    adam_params = dataset.SHO_fit_results(model = model_adam, phase_shift = np.pi / 2)

    trust_region_params = dataset.SHO_fit_results(model = model_trust_region, phase_shift = np.pi / 2)

    #BE_viz.SHO_hist([LSQF_params, adam_params, trust_region_params], SHO_ranges = BE_viz.SHO_ranges, filename=f"Figure_5_34_Histogram_comparison_{noise}_noise",)
    BE_viz.SHO_hist([LSQF_params, adam_params, trust_region_params], filename=f"Figure_5_34_Histogram_comparison_{noise}_noise",)

**Figure 5.34** Histogram of the fit results based on fits with noise 1 for the a-d. LSQF, e-h. neural network with ADAM, i-l. neural network with trust region optimizers.

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    LSQF_ = {'resampled': True,
                    'raw_format': 'complex',
                    'fitter': 'LSQF',
                    'scaled': False,
                    'output_shape': 'index',
                    'measurement_state': 'all',
                    'resampled_bins': 165,
                    'LSQF_phase_shift': 1.5707963267948966,
                    'NN_phase_shift': None,
                    'noise': noise}

    LSQF_Params = dataset.SHO_fit_results(state = LSQF_)

    adam_params = dataset.SHO_fit_results(model = model_adam, phase_shift = np.pi / 2)

    trust_region_params = dataset.SHO_fit_results(model = model_trust_region, phase_shift = np.pi / 2)

    BE_viz.SHO_switching_maps_test([LSQF_Params, adam_params, trust_region_params], filename=f"Figure_5_35_switching_maps_comparison_{noise}_noise", labels=["LSQF", "Adam", "SGD"])

**Figure 5.35** Visualize the switching based on the different models on the noisy data. 

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    BE_viz.SHO_fit_movie_images(noise = noise, 
                                models = [None, model_adam, model_trust_region],
                                scalebar_= True, 
                                basepath = "Movies/SHO_NN_compare",  
                                filename="SHO_NN_compare",
                                phase_shift = [None, np.pi / 2, np.pi / 2])

## Noise Level 6

### Scaling the Data

When training the neural network it is useful to scale the data. We apply a global scaler such that the spectrum have a mean of 0 and a standard deviation of 1.

#### Visualizing the Scaled Data


In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    # Sets the dataset
    noise = 6
    optimizer = "Adam"

    state = {"fitter": "LSQF", "resampled": True, "scaled": True, "label": "Scaled", "noise": noise}
    dataset.set_attributes(**state)

    BE_viz.nn_checker(state, filename=f"Figure_5_36_Scaled Raw Data_noise{noise}_optimizer_{optimizer}")

**Figure 5.36** Example visualization of the scaled, noisy data which is used for training.

### Extracts the Data and Models

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    # extracts the x and y data based on the noise
    X_data, Y_data = dataset.NN_data()

    # searches the trained models for the best model and loads it
    # (same architecture as trained by notebook 2_5)
    model_name_adam = basepath + "/" + results[(noise, "Adam")]['filename'].split("//")[-1]
    model_name_trust_region = basepath + "/" + results[(noise, "Trust Region CG")]['filename'].split("//")[-1]

    model_adam = load_nn_model(model_name_adam)

    model_trust_region = load_nn_model(model_name_trust_region)

### Evaluate the Fit Results

It is always recommended to validate that the autoencoder is working correctly. We can do this by visualizing the best, median, and worst fits.

We will assume that the autoencoder is working correctly and thus will not consider the test train split.

Note: we are comparing the autoencoder results to the original data, not the noisy data. 


In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    LSQF_ = {'resampled': True,
                    'raw_format': 'complex',
                    'fitter': 'LSQF',
                    'scaled': True,
                    'output_shape': 'index',
                    'measurement_state': 'all',
                    'resampled_bins': 165,
                    'LSQF_phase_shift': 1.5707963267948966,
                    'NN_phase_shift': None,
                    'noise': noise}


    BE_viz.MSE_compare(X_data_no_noise, [model_adam, model_trust_region, LSQF_], ["Adam", "Trust Region", "LSQF"])

#### Least Squares Fit

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:

    d1, d2, index1, mse1 = BE_viz.bmw_nn(
        X_data,
        prediction=LSQF_,
        out_state={"scaled": True, "raw_format": "complex"},
        returns=True,
        filename=f"Figure_5_37_NN_validation_noise_{noise}",
        compare_state = X_data_no_noise,
    )

**Figure 5.37** Visualization of the noisy (noise level 1) fit results from the least squares fitting algorithm shows the best, median, and worst fits.

#### Neural Network with Adam Optimizer

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    d1, d2, index1, mse1 = BE_viz.bmw_nn(
        X_data,
        prediction=model_adam,
        out_state={"scaled": True, "raw_format": "complex"},
        returns=True,
        filename=f"Figure_5_38_NN_validation_noise_{noise}_Adam",
        compare_state = X_data_no_noise,
    )

**Figure 5.38** Visualization of the noisy (noise level 0) fit results from the neural network trained with the Adam optimizers. We shows the best, median, and worst fits.

##### Neural Network with Trust Region Conjugate Gradient Optimizer

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    d1, d2, index1, mse1 = BE_viz.bmw_nn(
        X_data,
        prediction=model_trust_region,
        out_state={"scaled": True, "raw_format": "complex"},
        returns=True,
        filename=f"Figure_5_39_NN_validation_noise_{noise}_Trust_Region",
        compare_state = X_data_no_noise,
    )

**Figure 5.39** Visualization of the noisy fit results from the neural network trained with the Adam optimizers. We shows the best, median, and worst fits.

### Histogram of Fit Results

It is useful to view the histogram of the fitting results to apply any necessary phase shifts, and to see if the results are reasonable.


In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    # make all the histograms the same bin range.

    LSQF_ = {'resampled': True,
                    'raw_format': 'complex',
                    'fitter': 'LSQF',
                    'scaled': False,
                    'output_shape': 'index',
                    'measurement_state': 'all',
                    'resampled_bins': 165,
                    'LSQF_phase_shift': 1.5707963267948966,
                    'NN_phase_shift': None,
                    'noise': noise}

    dataset.set_attributes(**LSQF_)

    LSQF_params = dataset.SHO_fit_results(state = LSQF_)

    adam_params = dataset.SHO_fit_results(model = model_adam, phase_shift = np.pi / 2)

    trust_region_params = dataset.SHO_fit_results(model = model_trust_region, phase_shift = np.pi / 2)

    #BE_viz.SHO_hist([LSQF_params, adam_params, trust_region_params], SHO_ranges = BE_viz.SHO_ranges, filename=f"Figure_5_40_Histogram_comparison_{noise}_noise",)
    BE_viz.SHO_hist([LSQF_params, adam_params, trust_region_params], filename=f"Figure_5_40_Histogram_comparison_{noise}_noise",)

**Figure 5.40** Histogram of the fit results based on fits with noise 1 for the a-d. LSQF, e-h. neural network with ADAM, i-l. neural network with trust region optimizers.

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    LSQF_ = {'resampled': True,
                    'raw_format': 'complex',
                    'fitter': 'LSQF',
                    'scaled': False,
                    'output_shape': 'index',
                    'measurement_state': 'all',
                    'resampled_bins': 165,
                    'LSQF_phase_shift': 1.5707963267948966,
                    'NN_phase_shift': None,
                    'noise': noise}

    LSQF_Params = dataset.SHO_fit_results(state = LSQF_)

    adam_params = dataset.SHO_fit_results(model = model_adam, phase_shift = np.pi / 2)

    trust_region_params = dataset.SHO_fit_results(model = model_trust_region, phase_shift = np.pi / 2)

    BE_viz.SHO_switching_maps_test([LSQF_Params, adam_params, trust_region_params], filename=f"Figure_5_41_switching_maps_comparison_{noise}_noise", labels=["LSQF", "Adam", "SGD"])

**Figure 5.41** Visualize the switching based on the different models on the noisy data. 

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    BE_viz.SHO_fit_movie_images(noise = noise, 
                                models = [None, model_adam, model_trust_region],
                                scalebar_= True, 
                                basepath = "Movies/SHO_NN_compare",  
                                filename="SHO_NN_compare",
                                phase_shift = [None, np.pi / 2, np.pi / 2])

## Noise Level 7

### Scaling the Data

When training the neural network it is useful to scale the data. We apply a global scaler such that the spectrum have a mean of 0 and a standard deviation of 1.

#### Visualizing the Scaled Data


In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    # Sets the dataset
    noise = 7
    optimizer = "Adam"

    state = {"fitter": "LSQF", "resampled": True, "scaled": True, "label": "Scaled", "noise": noise}
    dataset.set_attributes(**state)

    BE_viz.nn_checker(state, filename=f"Figure_5_42_Scaled Raw Data_noise{noise}_optimizer_{optimizer}")

**Figure 5.42** Example visualization of the scaled, noisy data which is used for training.

### Extracts the Data and Models

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    # extracts the x and y data based on the noise
    X_data, Y_data = dataset.NN_data()

    # searches the trained models for the best model and loads it
    # (same architecture as trained by notebook 2_5)
    model_name_adam = basepath + "/" + results[(noise, "Adam")]['filename'].split("//")[-1]
    model_name_trust_region = basepath + "/" + results[(noise, "Trust Region CG")]['filename'].split("//")[-1]

    model_adam = load_nn_model(model_name_adam)

    model_trust_region = load_nn_model(model_name_trust_region)

### Evaluate the Fit Results

It is always recommended to validate that the autoencoder is working correctly. We can do this by visualizing the best, median, and worst fits.

We will assume that the autoencoder is working correctly and thus will not consider the test train split.

Note: we are comparing the autoencoder results to the original data, not the noisy data. 


In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    LSQF_ = {'resampled': True,
                    'raw_format': 'complex',
                    'fitter': 'LSQF',
                    'scaled': True,
                    'output_shape': 'index',
                    'measurement_state': 'all',
                    'resampled_bins': 165,
                    'LSQF_phase_shift': 1.5707963267948966,
                    'NN_phase_shift': None,
                    'noise': noise}


    BE_viz.MSE_compare(X_data_no_noise, [model_adam, model_trust_region, LSQF_], ["Adam", "Trust Region", "LSQF"])

#### Least Squares Fit

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:

    d1, d2, index1, mse1 = BE_viz.bmw_nn(
        X_data,
        prediction=LSQF_,
        out_state={"scaled": True, "raw_format": "complex"},
        returns=True,
        filename=f"Figure_5_43_NN_validation_noise_{noise}",
        compare_state = X_data_no_noise,
    )

**Figure 5.43** Visualization of the noisy (noise level 1) fit results from the least squares fitting algorithm shows the best, median, and worst fits.

#### Neural Network with Adam Optimizer

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    d1, d2, index1, mse1 = BE_viz.bmw_nn(
        X_data,
        prediction=model_adam,
        out_state={"scaled": True, "raw_format": "complex"},
        returns=True,
        filename=f"Figure_5_44_NN_validation_noise_{noise}_Adam",
        compare_state = X_data_no_noise,
    )

**Figure 5.44** Visualization of the noisy (noise level 0) fit results from the neural network trained with the Adam optimizers. We shows the best, median, and worst fits.

##### Neural Network with Trust Region Conjugate Gradient Optimizer

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    d1, d2, index1, mse1 = BE_viz.bmw_nn(
        X_data,
        prediction=model_trust_region,
        out_state={"scaled": True, "raw_format": "complex"},
        returns=True,
        filename=f"Figure_5_45_NN_validation_noise_{noise}_Trust_Region",
        compare_state = X_data_no_noise,
    )

**Figure 5.45** Visualization of the noisy fit results from the neural network trained with the Adam optimizers. We shows the best, median, and worst fits.

### Histogram of Fit Results

It is useful to view the histogram of the fitting results to apply any necessary phase shifts, and to see if the results are reasonable.


In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    # make all the histograms the same bin range.

    LSQF_ = {'resampled': True,
                    'raw_format': 'complex',
                    'fitter': 'LSQF',
                    'scaled': False,
                    'output_shape': 'index',
                    'measurement_state': 'all',
                    'resampled_bins': 165,
                    'LSQF_phase_shift': 1.5707963267948966,
                    'NN_phase_shift': None,
                    'noise': noise}

    dataset.set_attributes(**LSQF_)

    LSQF_params = dataset.SHO_fit_results(state = LSQF_)

    # adam_params = dataset.SHO_fit_results(model = model_adam, phase_shift = np.pi / 2)

    # trust_region_params = dataset.SHO_fit_results(model = model_trust_region, phase_shift = np.pi / 2)

    adam_params = dataset.SHO_fit_results(model = model_adam, phase_shift = np.pi / 2)

    trust_region_params = dataset.SHO_fit_results(model = model_trust_region, phase_shift = np.pi / 2)

    #BE_viz.SHO_hist([LSQF_params, adam_params, trust_region_params], SHO_ranges = BE_viz.SHO_ranges, filename=f"Figure_5_46_Histogram_comparison_{noise}_noise",)
    BE_viz.SHO_hist([LSQF_params, adam_params, trust_region_params], filename=f"Figure_5_46_Histogram_comparison_{noise}_noise",)

**Figure 5.46** Histogram of the fit results based on fits with noise 1 for the a-d. LSQF, e-h. neural network with ADAM, i-l. neural network with trust region optimizers.

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    LSQF_ = {'resampled': True,
                    'raw_format': 'complex',
                    'fitter': 'LSQF',
                    'scaled': False,
                    'output_shape': 'index',
                    'measurement_state': 'all',
                    'resampled_bins': 165,
                    'LSQF_phase_shift': 1.5707963267948966,
                    'NN_phase_shift': None,
                    'noise': noise}

    LSQF_Params = dataset.SHO_fit_results(state = LSQF_)

    adam_params = dataset.SHO_fit_results(model = model_adam, phase_shift = np.pi / 2)

    trust_region_params = dataset.SHO_fit_results(model = model_trust_region, phase_shift = np.pi / 2)

    BE_viz.SHO_switching_maps_test([LSQF_Params, adam_params, trust_region_params], filename=f"Figure_5_47_switching_maps_comparison_{noise}_noise", labels=["LSQF", "Adam", "SGD"])

**Figure 5.47** Visualize the switching based on the different models on the noisy data. 

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    BE_viz.SHO_fit_movie_images(noise = noise, 
                                models = [None, model_adam, model_trust_region],
                                scalebar_= True, 
                                basepath = "Movies/SHO_NN_compare",  
                                filename="SHO_NN_compare",
                                phase_shift = [None, np.pi / 2, np.pi / 2])

## Noise Level 8

### Scaling the Data

When training the neural network it is useful to scale the data. We apply a global scaler such that the spectrum have a mean of 0 and a standard deviation of 1.

#### Visualizing the Scaled Data


In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    # Sets the dataset
    noise = 8
    optimizer = "Adam"

    state = {"fitter": "LSQF", "resampled": True, "scaled": True, "label": "Scaled", "noise": noise}
    dataset.set_attributes(**state)

    BE_viz.nn_checker(state, filename=f"Figure_5_48_Scaled Raw Data_noise{noise}_optimizer_{optimizer}")

**Figure 5.48** Example visualization of the scaled, noisy data which is used for training.

### Extracts the Data and Models

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    # extracts the x and y data based on the noise
    X_data, Y_data = dataset.NN_data()

    # searches the trained models for the best model and loads it
    # (same architecture as trained by notebook 2_5)
    model_name_adam = basepath + "/" + results[(noise, "Adam")]['filename'].split("//")[-1]
    model_name_trust_region = basepath + "/" + results[(noise, "Trust Region CG")]['filename'].split("//")[-1]

    model_adam = load_nn_model(model_name_adam)

    model_trust_region = load_nn_model(model_name_trust_region)

### Evaluate the Fit Results

It is always recommended to validate that the autoencoder is working correctly. We can do this by visualizing the best, median, and worst fits.

We will assume that the autoencoder is working correctly and thus will not consider the test train split.

Note: we are comparing the autoencoder results to the original data, not the noisy data. 


In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    LSQF_ = {'resampled': True,
                    'raw_format': 'complex',
                    'fitter': 'LSQF',
                    'scaled': True,
                    'output_shape': 'index',
                    'measurement_state': 'all',
                    'resampled_bins': 165,
                    'LSQF_phase_shift': 1.5707963267948966,
                    'NN_phase_shift': None,
                    'noise': noise}


    BE_viz.MSE_compare(X_data_no_noise, [model_adam, model_trust_region, LSQF_], ["Adam", "Trust Region", "LSQF"])

#### Least Squares Fit

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:

    d1, d2, index1, mse1 = BE_viz.bmw_nn(
        X_data,
        prediction=LSQF_,
        out_state={"scaled": True, "raw_format": "complex"},
        returns=True,
        filename=f"Figure_5_49_NN_validation_noise_{noise}",
        compare_state = X_data_no_noise,
    )

**Figure 5.49** Visualization of the noisy (noise level 1) fit results from the least squares fitting algorithm shows the best, median, and worst fits.

#### Neural Network with Adam Optimizer

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    d1, d2, index1, mse1 = BE_viz.bmw_nn(
        X_data,
        prediction=model_adam,
        out_state={"scaled": True, "raw_format": "complex"},
        returns=True,
        filename=f"Figure_5_50_NN_validation_noise_{noise}_Adam",
        compare_state = X_data_no_noise,
    )

**Figure 5.50** Visualization of the noisy (noise level 0) fit results from the neural network trained with the Adam optimizers. We shows the best, median, and worst fits.

##### Neural Network with Trust Region Conjugate Gradient Optimizer

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    d1, d2, index1, mse1 = BE_viz.bmw_nn(
        X_data,
        prediction=model_trust_region,
        out_state={"scaled": True, "raw_format": "complex"},
        returns=True,
        filename=f"Figure_5_51_NN_validation_noise_{noise}_Trust_Region",
        compare_state = X_data_no_noise,
    )

**Figure 5.51** Visualization of the noisy fit results from the neural network trained with the Adam optimizers. We shows the best, median, and worst fits.

### Histogram of Fit Results

It is useful to view the histogram of the fitting results to apply any necessary phase shifts, and to see if the results are reasonable.


In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    # make all the histograms the same bin range.

    LSQF_ = {'resampled': True,
                    'raw_format': 'complex',
                    'fitter': 'LSQF',
                    'scaled': False,
                    'output_shape': 'index',
                    'measurement_state': 'all',
                    'resampled_bins': 165,
                    'LSQF_phase_shift': 1.5707963267948966,
                    'NN_phase_shift': None,
                    'noise': noise}

    dataset.set_attributes(**LSQF_)

    LSQF_params = dataset.SHO_fit_results(state = LSQF_)

    # adam_params = dataset.SHO_fit_results(model = model_adam, phase_shift = np.pi / 2)

    # trust_region_params = dataset.SHO_fit_results(model = model_trust_region, phase_shift = np.pi / 2)

    adam_params = dataset.SHO_fit_results(model = model_adam, phase_shift = np.pi / 2)

    trust_region_params = dataset.SHO_fit_results(model = model_trust_region, phase_shift = np.pi / 2)

    #BE_viz.SHO_hist([LSQF_params, adam_params, trust_region_params], SHO_ranges = BE_viz.SHO_ranges, filename=f"Figure_5_52_Histogram_comparison_{noise}_noise",)
    BE_viz.SHO_hist([LSQF_params, adam_params, trust_region_params], filename=f"Figure_5_52_Histogram_comparison_{noise}_noise",)

**Figure 5.52** Histogram of the fit results based on fits with noise 1 for the a-d. LSQF, e-h. neural network with ADAM, i-l. neural network with trust region optimizers.

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    LSQF_ = {'resampled': True,
                    'raw_format': 'complex',
                    'fitter': 'LSQF',
                    'scaled': False,
                    'output_shape': 'index',
                    'measurement_state': 'all',
                    'resampled_bins': 165,
                    'LSQF_phase_shift': 1.5707963267948966,
                    'NN_phase_shift': None,
                    'noise': noise}

    LSQF_Params = dataset.SHO_fit_results(state = LSQF_)

    adam_params = dataset.SHO_fit_results(model = model_adam, phase_shift = np.pi / 2)

    trust_region_params = dataset.SHO_fit_results(model = model_trust_region, phase_shift = np.pi / 2)

    BE_viz.SHO_switching_maps_test([LSQF_Params, adam_params, trust_region_params], filename=f"Figure_5_53_switching_maps_comparison_{noise}_noise", labels=["LSQF", "Adam", "SGD"])

**Figure 5.53** Visualize the switching based on the different models on the noisy data. 

In [ ]:
if QUICK_RUN:
    print("QUICK_RUN: requires full 0_5/2_5 outputs — run on Colab at full scale")
else:
    BE_viz.SHO_fit_movie_images(noise = noise, 
                                models = [None, model_adam, model_trust_region],
                                scalebar_= True, 
                                basepath = "Movies/SHO_NN_compare",  
                                filename="SHO_NN_compare",
                                phase_shift = [None, np.pi / 2, np.pi / 2])